<a href="https://colab.research.google.com/github/bala-baskar/LLM_foundations/blob/main/07_Reinforcement_Learning/Step_Size_Selection_for_Tile_Coding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implement a function that performs a single value function update using tile coding with a properly adjusted step size.

In tile coding, multiple overlapping tilings each activate exactly one tile for a given state. This means the total number of active features equals the number of tilings. If the step size is not adjusted to account for this, learning becomes unstable or overly slow.

Your function should:

1. Accept a base learning rate, the number of tilings, a weight vector, a list of active tile indices for the current state, and a target value.
2. Compute an adjusted step size that accounts for the number of tilings.
3. Compute the current value estimate as the sum of weights at the active tile indices.
4. Update only the weights corresponding to active tiles using the adjusted step size and the prediction error (target minus current estimate).
5. Compute the value estimate after the update.

Return a dictionary with keys: 'adjusted\_alpha' (the adjusted step size), 'prediction\_before' (value estimate before update), 'prediction\_after' (value estimate after update), and 'updated\_weights' (the full weight vector after update). All numerical values should be rounded to 4 decimal places.

## Example:

### Input:

```
base_alpha=0.1, n_tilings=4, weights=[0.0]*8, active_tiles=[0, 2, 4, 6], target=2.0
```

### Output:

```
{'adjusted_alpha': 0.025, 'prediction_before': 0.0, 'prediction_after': 0.2, 'updated_weights': [0.05, 0.0, 0.05, 0.0, 0.05, 0.0, 0.05, 0.0]}
```

### Reasoning:

With 4 tilings, the adjusted step size is 0.1 / 4 = 0.025. The current prediction is the sum of weights at active indices \[0, 2, 4, 6\], all zero, so prediction\_before = 0.0. The error is 2.0 - 0.0 = 2.0. Each of the 4 active tiles receives an update of 0.025 \* 2.0 = 0.05. The new prediction is 4 \* 0.05 = 0.2. Note the effective change in prediction is base\_alpha \* error = 0.1 \* 2.0 = 0.2, which shows the adjustment makes the effective learning rate behave as the base rate intended.

Learn About topic

## Step-Size Selection for Tile Coding

### Why Step-Size Adjustment Matters

In tile coding, a state is represented by activating exactly one tile from each of the $n$ overlapping tilings. The value function is approximated as the sum of weights at all active tiles:

$\hat{v}(s, \mathbf{w}) = \sum_{i \in \mathcal{A}(s)} w_i$

where $\mathcal{A}(s)$ is the set of active tile indices for state $s$, and $|\mathcal{A}(s)| = n$ (the number of tilings).

### The Problem with Unadjusted Step Sizes

The standard semi-gradient update rule for linear function approximation is:

$w_i \leftarrow w_i + \alpha \, \delta \, x_i(s)$

where $\delta = G_t - \hat{v}(s, \mathbf{w})$ is the prediction error and $x_i(s) \in \{0, 1\}$ is the feature value. Since $n$ features are active simultaneously, the total change in the prediction after one update is:

$\Delta \hat{v} = \sum_{i \in \mathcal{A}(s)} \alpha \, \delta \, x_i(s) = n \cdot \alpha \cdot \delta$

This means the effective learning rate is $n$ times larger than the nominal $\alpha$. With many tilings, this can cause overshooting and instability.

### The Adjustment Rule

To ensure the effective learning rate matches the desired base rate $\alpha_0$, we divide by the number of tilings:

$\alpha = \frac{\alpha_0}{n}$

With this adjustment, the total prediction change becomes:

$\Delta \hat{v} = n \cdot \frac{\alpha_0}{n} \cdot \delta = \alpha_0 \cdot \delta$

This gives the intended behavior regardless of how many tilings are used.

### Special Case: One-Shot Learning

When $\alpha_0 = 1$, the adjusted step size is $\alpha = 1/n$. In this case, a single update changes the prediction to exactly match the target:

$\hat{v}_{\text{new}} = \hat{v}_{\text{old}} + 1.0 \cdot (\text{target} - \hat{v}_{\text{old}}) = \text{target}$

This is the most aggressive setting that still avoids overshooting on a single sample. In practice, smaller values of $\alpha_0$ (e.g., 0.1) are used because the agent visits many states and must balance learning across them.

### General Principle

More generally, when $k$ features are active for any given state, the step size should be set proportional to $1/k$ to maintain a consistent effective learning rate. For tile coding, $k = n$ is constant for all states, making the adjustment straightforward.


In [18]:
import numpy as np

def tile_coding_step(base_alpha: float, n_tilings: int, weights: list, active_tiles: list, target: float) -> dict:
    """
    Perform a single value function update using tile coding with adjusted step size.

    Args:
        base_alpha: The base learning rate before adjustment
        n_tilings: Number of tilings used in the tile coding scheme
        weights: List of current weight values for all tiles
        active_tiles: List of indices of tiles active for the current state
        target: The target value (e.g., a return or TD target)

    Returns:
        Dictionary with 'adjusted_alpha', 'prediction_before',
        'prediction_after', and 'updated_weights'
    """
    output = dict()
    output['adjusted_alpha'] = round(base_alpha / n_tilings,4)
    active_weights = [val for id, val in enumerate(weights) if id in active_tiles]
    output['prediction_before'] = round(np.sum(active_weights),4)
    error = target - output['prediction_before']
    output['prediction_after'] = round(output['prediction_before'] + error * output['adjusted_alpha']* n_tilings,4)
    updated_weights = weights.copy()
    for idx in active_tiles:
        updated_weights[idx] = round(error * output['adjusted_alpha'] + weights[idx],4)

    output['updated_weights'] = updated_weights
    return output


In [19]:
result = tile_coding_step(0.2, 2, [0.5, 0.3, 0.1, 0.4], [0, 3], 1.0)
print(result)

{'adjusted_alpha': 0.1, 'prediction_before': np.float64(0.9), 'prediction_after': np.float64(0.92), 'updated_weights': [np.float64(0.51), 0.3, 0.1, np.float64(0.41)]}


In [20]:
result = tile_coding_step(0.5, 4, [1.0, 1.0, 1.0, 1.0], [0, 1, 2, 3], 2.0)
print(result)

{'adjusted_alpha': 0.125, 'prediction_before': np.float64(4.0), 'prediction_after': np.float64(3.0), 'updated_weights': [np.float64(0.75), np.float64(0.75), np.float64(0.75), np.float64(0.75)]}
